In [2]:
import pandas as pd
po_lines = pd.read_csv('../data/raw/purchase_orders_lines.csv')
print(po_lines.head())


       po_id  line_number product_id  quantity  unit_cost  gst_rate  \
0  PO-141186            1       P004       234   34709.54        28   
1  PO-141186            2       P022        79   12317.84        28   
2  PO-141186            3       P030        46   13840.45        28   
3  PO-141186            4       P001       293   16988.41        28   
4  PO-141186            5       P014       126   65443.06        28   

   line_total    gst_amount  line_grand_total  
0  8122032.36  2.274169e+06      1.039620e+07  
1   973109.36  2.724706e+05      1.245580e+06  
2   636660.70  1.782650e+05      8.149257e+05  
3  4977604.13  1.393729e+06      6.371333e+06  
4  8245825.56  2.308831e+06      1.055466e+07  


In [6]:
print("=== INFO ===")
print(po_lines.info())

print("\n=== MISSING VALUES ===")
print(po_lines.isnull().sum())

print("\n=== SHAPE ===")
print(po_lines.shape)

print("\n=== SAMPLE DATA ===")
po_lines.head()

=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 155495 entries, 0 to 155494
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   po_id             155495 non-null  str    
 1   line_number       155495 non-null  int64  
 2   product_id        155495 non-null  str    
 3   quantity          155495 non-null  int64  
 4   unit_cost         155495 non-null  float64
 5   gst_rate          155495 non-null  int64  
 6   line_total        155495 non-null  float64
 7   gst_amount        155495 non-null  float64
 8   line_grand_total  155495 non-null  float64
dtypes: float64(4), int64(3), str(2)
memory usage: 10.7 MB
None

=== MISSING VALUES ===
po_id               0
line_number         0
product_id          0
quantity            0
unit_cost           0
gst_rate            0
line_total          0
gst_amount          0
line_grand_total    0
dtype: int64

=== SHAPE ===
(155495, 9)

=== SAMPLE DATA ===


,po_id,line_number,product_id,quantity,unit_cost,gst_rate,line_total,gst_amount,line_grand_total
0,PO-141186,1,P004,234,34709.54,28,8122032.36,2.274169e+06,1.039620e+07
1,PO-141186,2,P022,79,12317.84,28,973109.36,2.724706e+05,1.245580e+06
2,PO-141186,3,P030,46,13840.45,28,636660.70,1.782650e+05,8.149257e+05
3,PO-141186,4,P001,293,16988.41,28,4977604.13,1.393729e+06,6.371333e+06
4,PO-141186,5,P014,126,65443.06,28,8245825.56,2.308831e+06,1.055466e+07


In [7]:
text_cols = [
    'po_id', 'product_id',
]

for col in text_cols:
    po_lines[col] = po_lines[col].str.upper()

In [8]:
numeric_cols = [
    'line_number',
    'quantity',
    'unit_cost',
    'gst_rate',
    'line_total',
    'gst_amount',
    'line_grand_total'
]

po_lines[numeric_cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 155495 entries, 0 to 155494
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   line_number       155495 non-null  int64  
 1   quantity          155495 non-null  int64  
 2   unit_cost         155495 non-null  float64
 3   gst_rate          155495 non-null  int64  
 4   line_total        155495 non-null  float64
 5   gst_amount        155495 non-null  float64
 6   line_grand_total  155495 non-null  float64
dtypes: float64(4), int64(3)
memory usage: 8.3 MB


In [11]:
po_lines['calculated_line_total'] = (
    po_lines['quantity'] * po_lines['unit_cost']
)


po_lines['calculated_gst'] = (
    po_lines['calculated_line_total']
    * (po_lines['gst_rate'] / 100)
)

po_lines['calculated_grand_total'] = (
    po_lines['calculated_line_total']
    + po_lines['calculated_gst']
)

In [12]:
po_lines['line_total_variance'] = (
    po_lines['line_total']
    - po_lines['calculated_line_total']
)

po_lines['gst_variance'] = (
    po_lines['gst_amount']
    - po_lines['calculated_gst']
)

po_lines['grand_total_variance'] = (
    po_lines['line_grand_total']
    - po_lines['calculated_grand_total']
)

In [14]:
po_lines['actual_gst_pct'] = (    po_lines['gst_amount'] / po_lines['line_total']) * 100

median_value = po_lines['line_grand_total'].median()

po_lines['high_value_line'] = (
    po_lines['line_grand_total'] > median_value
).astype(int)


median_value = po_lines['line_grand_total'].median()

po_lines['high_value_line'] = (
    po_lines['line_grand_total'] > median_value
).astype(int)

In [18]:
duplicates = po_lines.duplicated(
    subset=['po_id', 'line_number']
).sum()

print("Duplicate PO Lines:", duplicates)

po_lines['product_id'].nunique()

po_lines[numeric_cols].describe()



Duplicate PO Lines: 0


,line_number,quantity,unit_cost,gst_rate,line_total,gst_amount,line_grand_total
count,155495.000000,155495.000000,155495.000000,155495.000000,1.554950e+05,1.554950e+05,1.554950e+05
mean,4.670092,159.926146,13922.761264,23.680890,2.229152e+06,6.113658e+05,2.840518e+06
std,2.932263,81.116946,19721.767911,4.953438,3.713297e+06,1.044680e+06,4.757601e+06
min,1.000000,20.000000,186.050000,18.000000,3.763600e+03,6.774480e+02,4.441048e+03
25%,2.000000,90.000000,1034.540000,18.000000,1.512209e+05,2.819062e+04,1.801162e+05
50%,4.000000,160.000000,7324.630000,28.000000,8.338085e+05,2.041258e+05,1.037774e+06
75%,7.000000,230.000000,18321.315000,28.000000,2.543595e+06,7.107491e+05,3.249139e+06
max,12.000000,300.000000,107090.250000,28.000000,3.210214e+07,8.988601e+06,4.109075e+07


In [19]:
top_products = (
    po_lines.groupby('product_id')['line_grand_total']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products

product_id
P015    9.309059e+10
P014    6.782379e+10
P021    3.669053e+10
P004    3.139966e+10
P019    3.078018e+10
P010    2.859236e+10
P011    2.208656e+10
P001    1.972090e+10
P030    1.723771e+10
P022    1.353251e+10
Name: line_grand_total, dtype: float64

In [20]:
po_lines['line_grand_total'].sum()

po_lines['line_grand_total'].mean()

po_lines['quantity'].mean()


np.float64(159.92614553522623)

In [21]:
po_lines.drop_duplicates(inplace=True)

print(po_lines.shape)

(155495, 17)


In [22]:
po_lines.to_csv('../data/processed/purchase_orders_lines.csv',
    index=False
)

print("Processed purchase_orders_lines.csv saved successfully!")

Processed purchase_orders_lines.csv saved successfully!
